In [4]:
!pip install pyspark --quiet
print('pyspark installation complete')

pyspark installation complete


In [5]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import year,month,to_date,col,round as spark_round
import matplotlib.pyplot as plt
import pandas as pd
import warnings
warnings.filterwarnings('ignore')
spark = SparkSession.builder \
    .appName('Day4_bd_sales') \
    .config('spark.sql.adaptive.enables','true') \
    .getOrCreate()
print(f'spark version : {spark.version}')
print(f'SparkSession : ACTIVE')
print(f'application name:{spark.conf.get("spark.app.name")}')


spark version : 4.0.2
SparkSession : ACTIVE
application name:Day4_bd_sales


In [8]:
from google.colab import files
uploaded = files.upload()

Saving large_sales_data.csv to large_sales_data.csv


In [6]:
df_bronze = spark.read \
   .option('header', 'true') \
   .option('inferSchema', 'true') \
   .csv('large_sales_data.csv')
print('=== BRONZEN LAYER - Raw Data ===')
print(f'Rows : {df_bronze.count()}')
print(f'Columns : {len(df_bronze.columns)}')
print()
df_bronze.printSchema()

=== BRONZEN LAYER - Raw Data ===
Rows : 5000
Columns : 13

root
 |-- order_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- revenue: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- city: string (nullable = true)
 |-- region: string (nullable = true)
 |-- sales_rep: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- order_status: string (nullable = true)



In [7]:
print('first 5 rows')
df_bronze.show(5, truncate=False)
print('\nBasic statistics for numeric column:')
df_bronze.select('quantity','unit_price','revenue').describe().show()

first 5 rows
+--------+-------------+----------+-----------+--------+----------+-------+----------+---------+------+-----------+----------------+------------+
|order_id|customer_name|product   |category   |quantity|unit_price|revenue|order_date|city     |region|sales_rep  |payment_method  |order_status|
+--------+-------------+----------+-----------+--------+----------+-------+----------+---------+------+-----------+----------------+------------+
|1001    |Sneha Reddy  |Monitor   |Electronics|12      |22000     |264000 |2023-05-21|Mumbai   |West  |Meera Patel|UPI             |Delivered   |
|1002    |Ramesh Kumar |Printer   |Electronics|10      |12000     |120000 |2023-08-05|Delhi    |North |Anil Sharma|Credit Card     |Shipped     |
|1003    |Rahul Mishra |Mouse     |Accessories|10      |800       |8000   |2023-01-14|Ahmedabad|West  |Meera Patel|Cash on Delivery|Shipped     |
|1004    |Suresh Rao   |Tablet    |Electronics|5       |32000     |160000 |2023-01-04|Surat    |West  |Ravi Kum

In [21]:
df_bronze.write \
   .mode('overwrite') \
   .parquet('sales_bronze.parquet')
print('bronze parquet saved:sales_bronze.parquet')
import os
def get_dir_size(path):
  """Get total size of a file or directory in KB."""
  if os.path.isfile(path):
    return os.path.getsize(path)  /1024
  total = 0
  for dirpath, dirnames, filenames in os.walk(path):
    for f in filenames:
      total += os.path.getsize(os.path.join(dirpath,f))
  return total  /1024
csv_size =get_dir_size('large_sales_data.csv')
parquet_size =get_dir_size('sales_bronze.parquet')
reduction =(1 - parquet_size/csv_size) *100
print(f'nCSV size : (csv_size:.1f) KB')
print(f'Parquet size : (parquet_size:.1f) KB')
print(f'reduction : (reduction:.1f) % smaller')
print(f'\nAt 1 TB scale: CSV=1000 GB -> Parquet={1000*(1-reduction/100):.0f}GB')


bronze parquet saved:sales_bronze.parquet
nCSV size : (csv_size:.1f) KB
Parquet size : (parquet_size:.1f) KB
reduction : (reduction:.1f) % smaller

At 1 TB scale: CSV=1000 GB -> Parquet=104GB


In [23]:
df_silver = df_bronze \
  .dropDuplicates() \
  .dropna(subset=['order_id','product','revenue'])
df_silver = df_silver.withColumn(
    'order_date',
    to_date(col('order_date'), 'yyyy-MM-dd')
)
df_silver = df_silver \
     .withColumn('order_year', year(col('order_date'))) \
     .withColumn('order_month', month(col('order_date')))
df_silver = df_silver.withColumn(
    'revenue_category',
    F.when(col('revenue') > 4000, 'High')
    .when(col('revenue') > 1000, 'Medium')
    .otherwise('Low')
)
print(f'silver layer rows : {df_silver.count()}')
print('New columns added: order_year, order_month, revenue_category')
df_silver.select('product','revenue','order_year','order_month','revenue_category')

silver layer rows : 5000
New columns added: order_year, order_month, revenue_category


DataFrame[product: string, revenue: int, order_year: int, order_month: int, revenue_category: string]

In [24]:
df_silver.write \
   .mode('overwrite') \
   .parquet('sales_silver.parquet')
print('silver parquet saved: sales_silver.parquet')
print(f'silver size: {get_dir_size("sales_silver.parquet"):.1f} KB')
df_verify = spark.read.parquet('sales_silver.parquet')
print('n=== Verify Silver Layer ===')
print(f'read-back rows: {df_verify.count()} (should match silver count)')
df_verify.printSchema()


silver parquet saved: sales_silver.parquet
silver size: 59.5 KB
n=== Verify Silver Layer ===
read-back rows: 5000 (should match silver count)
root
 |-- order_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- revenue: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- city: string (nullable = true)
 |-- region: string (nullable = true)
 |-- sales_rep: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_year: integer (nullable = true)
 |-- order_month: integer (nullable = true)
 |-- revenue_category: string (nullable = true)

